# Real-World — Hidden Seasonality Detection in Time Series

We synthesize a business KPI with weekly seasonality plus noise. We then:
- Estimate the dominant period via FFT (baseline)
- Build a matching `PeriodicState` and show QFT peak structure as an explanatory tool

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import rfft, rfftfreq
from quantum_hybrid_system import PeriodicState

# Synthesize daily data with weekly seasonality (period ~7), noise, and drift
rng = np.random.default_rng(42)
days = 180
t = np.arange(days)
signal = 10 + 2*np.sin(2*np.pi*t/7) + 0.01*t + rng.normal(0, 0.8, size=days)

# FFT baseline
yf = np.abs(rfft(signal - signal.mean()))
xf = rfftfreq(days, d=1.0)
dominant = xf[1:][yf[1:].argmax()]
period_est = 1/dominant if dominant>0 else np.nan
print("FFT dominant frequency:", dominant, " -> period ~", period_est)

plt.figure()
plt.plot(t, signal)
plt.title("Synthetic KPI (seasonality + noise)")
plt.xlabel("day")
plt.ylabel("value")
plt.show()

# Map to a nearest power-of-two register size and compare with PeriodicState QFT peaks
n = 10  # 1024 bins
N = 2**n
r = int(round(period_est)) if np.isfinite(period_est) else 7
r = max(2, min(r, 64))
ps = PeriodicState(num_qubits=n, period=r)
shots = 4000
samples = ps.measure(num_shots=shots, use_qft=True)

# Show coarse histogram of QFT samples
bins = 64
hist = np.zeros(bins, dtype=int)
for s in samples:
    hist[(s * bins) // N] += 1

plt.figure()
plt.bar(np.arange(bins), hist)
plt.title(f"Analytic QFT sampling histogram (r≈{r})")
plt.xlabel("coarse frequency bin")
plt.ylabel("counts")
plt.show()